# 서울시 따릉이 스테이션별 대여 수요 예측 파이프라인

## 환경 및 데이터 로드 (Data Pipeline Setup)

### 환경 설정 및 라이브러리 임포트

In [ ]:
# ==========================================
# 라이브러리 임포트
# ==========================================
import os
import sys
import gc
import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from datetime import datetime
from functools import reduce
from dotenv import load_dotenv

import geopandas as gpd
from shapely import wkt
from IPython.display import display
from sqlalchemy import create_engine, text

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


# ==========================================
# 환경 설정 및 경로 로드
# ==========================================
sys.path.append(os.path.dirname(os.getcwd()))
load_dotenv()


# ==========================================
# 시각화 및 전역 환경 설정
# ==========================================
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams['font.family'] = 'Malgun Gothic'

SEED = 42
np.random.seed(SEED)

print("========== 데이터 분석 환경 설정 완료 ==========")

### 데이터베이스 연결 및 원본 데이터 로드

In [ ]:
# ==========================================
# 데이터베이스 연결 설정
# ==========================================
DB_USER = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "password")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_NAME = os.getenv("DB_NAME", "seoul_bike")

DATABASE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

with engine.connect() as conn:
    if conn.execute(text("SELECT 1")).scalar() == 1:
        print("========== 데이터베이스 연결 성공 ==========")
    else:
        print("========== 데이터베이스 연결 실패 ==========")


# ==========================================
# 데이터 로드
# ==========================================
target_tables = [
    "hourly_air_2024", "hourly_precip_2024", "hourly_snow_2024", "hourly_temp_2024",
    "rt_air", "rt_weather", "infra_business", "infra_park", "infra_river",
    "infra_school", "infra_univ", "infra_subway", "pop_flow_2024", "pop_living_2024",
    "korea_holidays", "station_loc", "rent_history_2023"
]

# 테이블 데이터 로드 및 딕셔너리 관리
df_dict = {}
for table in target_tables:
    try:
        df_dict[table] = pd.read_sql_table(table, con=engine)
    except Exception as e:
        print(f"[{table}] 로드 실패: {e}")

# 개별 변수 할당
hourly_air_2024_df    = df_dict["hourly_air_2024"]
hourly_precip_2024_df = df_dict["hourly_precip_2024"]
hourly_snow_2024_df   = df_dict["hourly_snow_2024"]
hourly_temp_2024_df   = df_dict["hourly_temp_2024"]
rt_air_df             = df_dict["rt_air"]
rt_weather_df         = df_dict["rt_weather"]
infra_business_df     = df_dict["infra_business"]
infra_park_df         = df_dict["infra_park"]
infra_river_df        = df_dict["infra_river"]
infra_school_df       = df_dict["infra_school"]
infra_univ_df         = df_dict["infra_univ"]
infra_subway_df       = df_dict["infra_subway"]
pop_flow_2024_df      = df_dict["pop_flow_2024"]
pop_living_2024_df    = df_dict["pop_living_2024"]
korea_holidays_df     = df_dict["korea_holidays"]
station_loc_df        = df_dict["station_loc"]
rent_history_2023_df  = df_dict["rent_history_2023"]

# 따릉이 이력 데이터 로드 (청크 단위 처리)
table_name = "rent_history_2024"
chunk_size = 100000
chunk_iterator = pd.read_sql_table(table_name, con=engine, chunksize=chunk_size)
rent_history_2024_df = pd.concat([chunk for chunk in chunk_iterator], ignore_index=True)
df_dict[table_name] = rent_history_2024_df


# ==========================================
# 데이터 로드 요약 출력
# ==========================================
summary_data = [
    {"Table Name": name, "Row Count": len(df)}
    for name, df in df_dict.items()
]

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values(by="Row Count", ascending=False).reset_index(drop=True)
summary_df["Row Count"] = summary_df["Row Count"].apply(lambda x: f"{x:,}")

display(summary_df)

## 데이터 전처리 (Data Preprocessing)

### 따릉이 대여소 위치, 환경, 인구, 인프라 데이터 전처리

In [ ]:
# ==========================================
# 대여소 위치 좌표 보정
# ==========================================
HARDCODED_COORDS = {
    'ST-1066': (37.55290, 126.83650), 'ST-1068': (37.54897, 126.84852),
    'ST-1073': (37.50325, 127.12782), 'ST-1074': (37.49830, 127.13454),
    'ST-1090': (37.48083, 127.12933), 'ST-1091': (37.50743, 127.10123),
    'ST-1255': (37.56847, 126.84803), 'ST-1318': (37.53424, 126.89736),
    'ST-1412': (37.50383, 127.13876), 'ST-1415': (37.48161, 127.14361),
    'ST-2':    (37.55088, 126.91039), 'ST-415':  (37.51980, 126.88937),
    'ST-423':  (37.52784, 126.92873), 'ST-989':  (37.54955, 126.91071)
}

station_loc_df['station_id'] = station_loc_df['station_id'].astype(str).str.strip()

for st_id, (lat, lon) in HARDCODED_COORDS.items():
    mask = station_loc_df['station_id'] == st_id
    station_loc_df.loc[mask, ['lat', 'lon']] = [lat, lon]


# ==========================================
# 환경 데이터 전처리
# ==========================================
def preprocess_env(df, col_name):
    df = df.copy()
    if 'id' in df.columns:
        df = df.drop(columns=['id'])

    df['measure_date'] = pd.to_datetime(df['measure_date'])
    df.replace([-9, -9.0], np.nan, inplace=True)

    return df.set_index('measure_date').groupby('region_name')[col_name].resample('1h').mean().reset_index()

env_dfs = [
    preprocess_env(hourly_air_2024_df, 'pm10'),
    preprocess_env(hourly_temp_2024_df, 'temperature'),
    preprocess_env(hourly_precip_2024_df, 'precipitation'),
    preprocess_env(hourly_snow_2024_df, 'snowfall')
]

env_master_2024_df = reduce(lambda l, r: pd.merge(l, r, on=['measure_date', 'region_name'], how='outer'), env_dfs)
env_master_2024_df = env_master_2024_df.sort_values(by=['region_name', 'measure_date']).reset_index(drop=True)

# 결측치 보간 처리
env_master_2024_df['temperature'] = env_master_2024_df.groupby('region_name')['temperature'].transform(lambda x: x.interpolate(method='linear').ffill().bfill())
env_master_2024_df['pm10'] = env_master_2024_df.groupby('region_name')['pm10'].transform(lambda x: x.interpolate(method='linear').ffill().bfill())
env_master_2024_df['precipitation'] = env_master_2024_df['precipitation'].fillna(0)
env_master_2024_df['snowfall'] = env_master_2024_df['snowfall'].fillna(0)


# ==========================================
# 인프라 공간 데이터 변환 및 피처 연산
# ==========================================
def preprocess_gdf(df, wkt_col=None):
    if df.empty:
        return gpd.GeoDataFrame()

    if wkt_col:
        df = df.dropna(subset=[wkt_col]).copy()
        df['geometry'] = df[wkt_col].apply(lambda x: wkt.loads(str(x)) if pd.notna(x) and str(x) != 'None' else None)
        df = df.dropna(subset=['geometry'])
        gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")
    else:
        lat_col = 'latitude' if 'latitude' in df.columns else 'lat' if 'lat' in df.columns else None
        lon_col = 'longitude' if 'longitude' in df.columns else 'lon' if 'lon' in df.columns else 'lot' if 'lot' in df.columns else None

        if not lat_col or not lon_col:
            return gpd.GeoDataFrame()

        df = df.dropna(subset=[lat_col, lon_col]).copy()
        gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lon_col].astype(float), df[lat_col].astype(float)), crs="EPSG:4326")

    return gdf.to_crs(epsg=5179)[~gdf.is_empty]

def count_infra(tgt, src, r, col):
    if src is None or src.empty:
        return pd.Series(0, index=tgt.index, name=col)

    buf = tgt.copy()
    buf['geometry'] = buf.geometry.buffer(r)
    joined = gpd.sjoin(buf, src, how='left', predicate='intersects')
    return joined.groupby(joined.index)['index_right'].count().rename(col)

def cal_nearest_infra(tgt, src, col):
    if src is None or src.empty:
        return pd.DataFrame({col: [np.nan] * len(tgt)}, index=tgt.index)

    nearest = gpd.sjoin_nearest(tgt, src, distance_col=col)
    return nearest[~nearest.index.duplicated(keep='first')][[col]]

# 인프라 원본 데이터 전처리
station_loc_gdf = preprocess_gdf(station_loc_df)
infra_park_df['lon'] = infra_park_df['xcrd_g'].fillna(infra_park_df['xcrd'])
infra_park_df['lat'] = infra_park_df['ycrd_g'].fillna(infra_park_df['ycrd'])
infra_subway_df['lon'] = pd.to_numeric(infra_subway_df.get('lon', infra_subway_df.get('lot')), errors='coerce')

# GeoDataFrame 생성
infra_park_gdf = preprocess_gdf(infra_park_df)
infra_river_gdf = preprocess_gdf(infra_river_df, wkt_col='geom_wkt')
infra_subway_gdf = preprocess_gdf(infra_subway_df)
infra_business_gdf = preprocess_gdf(infra_business_df)
infra_edu_gdf = pd.concat([preprocess_gdf(infra_school_df), preprocess_gdf(infra_univ_df)], ignore_index=True)

# 피처 연산 적용
infra_master_df = station_loc_gdf.copy()
infra_master_df['subway_cnt_300m'] = count_infra(infra_master_df, infra_subway_gdf, 300, 'subway_cnt_300m')
infra_master_df['biz_cnt_300m'] = count_infra(infra_master_df, infra_business_gdf, 300, 'biz_cnt_300m')
infra_master_df['edu_cnt_500m'] = count_infra(infra_master_df, infra_edu_gdf, 500, 'edu_cnt_500m')
infra_master_df['park_cnt_500m'] = count_infra(infra_master_df, infra_park_gdf, 500, 'park_cnt_500m')
infra_master_df['river_cnt_1km'] = count_infra(infra_master_df, infra_river_gdf, 1000, 'river_cnt_1km')

infra_master_df = infra_master_df.join(cal_nearest_infra(infra_master_df, infra_subway_gdf, 'dist_subway'))
infra_master_df = infra_master_df.join(cal_nearest_infra(infra_master_df, infra_river_gdf, 'dist_river'))
infra_master_df = infra_master_df.drop(columns=['geometry'])

print("========== 전처리 및 병합 완료 ==========")

### 인구 데이터 전처리

In [ ]:
# ==========================================
# 생활인구 데이터 전처리
# ==========================================
district_map = {'11500': '강서구', '11560': '영등포구', '11440': '마포구', '11710': '송파구'}

pop_living_2024_df['district_name'] = pop_living_2024_df['adstrd_code_se'].map(district_map)
pop_living_2024_df.rename(columns={'tot_lvpop_co': 'lvgpop_tot'}, inplace=True)

# 연령대별 그룹화
age_groups = {
    'lvgpop_10s': ['male_f10t14_lvpop_co', 'male_f15t19_lvpop_co', 'female_f10t14_lvpop_co', 'female_f15t19_lvpop_co'],
    'lvgpop_20s': ['male_f20t24_lvpop_co', 'male_f25t29_lvpop_co', 'female_f20t24_lvpop_co', 'female_f25t29_lvpop_co'],
    'lvgpop_30s': ['male_f30t34_lvpop_co', 'male_f35t39_lvpop_co', 'female_f30t34_lvpop_co', 'female_f35t39_lvpop_co'],
    'lvgpop_40s': ['male_f40t44_lvpop_co', 'male_f45t49_lvpop_co', 'female_f40t44_lvpop_co', 'female_f45t49_lvpop_co'],
    'lvgpop_50s': ['male_f50t54_lvpop_co', 'male_f55t59_lvpop_co', 'female_f50t54_lvpop_co', 'female_f55t59_lvpop_co'],
    'lvgpop_60up': ['male_f60t64_lvpop_co', 'male_f65t69_lvpop_co', 'male_f70t74_lvpop_co',
                    'female_f60t64_lvpop_co', 'female_f65t69_lvpop_co', 'female_f70t74_lvpop_co']
}

for col, src_cols in age_groups.items():
    pop_living_2024_df[col] = pop_living_2024_df[src_cols].sum(axis=1)

# 병합용 날짜 및 시간 키 생성
pop_living_2024_df = pop_living_2024_df[['stdr_de_id', 'tmzon_pd_se', 'district_name', 'lvgpop_tot'] + list(age_groups.keys())].copy()
pop_living_2024_df['date_str'] = pop_living_2024_df['stdr_de_id'].astype(str)
pop_living_2024_df['hour_str'] = pop_living_2024_df['tmzon_pd_se'].astype(str).str.zfill(2)


# ==========================================
# 인구 마스터 뼈대 생성
# ==========================================
date_rng = pd.date_range(start='2024-01-01', end='2024-12-31', freq='h')
districts = ['강서구', '영등포구', '마포구', '송파구']

pop_master_list = []
for dist in districts:
    df_temp = pd.DataFrame(date_rng, columns=['datetime'])
    df_temp['district_name'] = dist
    df_temp['date_str'] = df_temp['datetime'].dt.strftime('%Y%m%d')
    df_temp['hour_str'] = df_temp['datetime'].dt.strftime('%H')
    df_temp['weekday'] = df_temp['datetime'].dt.weekday
    df_temp['quarter_str'] = '2024' + df_temp['datetime'].dt.quarter.astype(str)
    pop_master_list.append(df_temp)

pop_master_2024_df = pd.concat(pop_master_list, ignore_index=True)


# ==========================================
# 유동인구 데이터 병합 및 벡터 연산
# ==========================================
pop_flow_2024_df = pd.merge(
    pop_master_2024_df, pop_flow_2024_df,
    left_on=['quarter_str', 'district_name'], right_on=['stdr_yyqu_cd', 'signgu_cd_nm'],
    how='left'
)

# 시간 및 요일별 인구 분배 가중치 계산
h = pop_flow_2024_df['hour_str'].astype(int)
wd = pop_flow_2024_df['weekday']

div = np.select([h.isin(range(6, 11)), h.isin(range(11, 17)), h.isin(range(17, 21)), h.isin(range(21, 24))],
                [5, 3, 4, 3], default=6)

time_pop = np.select(
    [h.isin(range(0, 6)), h.isin(range(6, 11)), h.isin(range(11, 14)), h.isin(range(14, 17)), h.isin(range(17, 21))],
    [pop_flow_2024_df['tmzon_00_06_flpop_co'], pop_flow_2024_df['tmzon_06_11_flpop_co'],
     pop_flow_2024_df['tmzon_11_14_flpop_co'], pop_flow_2024_df['tmzon_14_17_flpop_co'],
     pop_flow_2024_df['tmzon_17_21_flpop_co']],
    default=pop_flow_2024_df['tmzon_21_24_flpop_co']
)

day_pop = np.select(
    [wd == 0, wd == 1, wd == 2, wd == 3, wd == 4, wd == 5, wd == 6],
    [pop_flow_2024_df['mon_flpop_co'], pop_flow_2024_df['tues_flpop_co'], pop_flow_2024_df['wed_flpop_co'],
     pop_flow_2024_df['thur_flpop_co'], pop_flow_2024_df['fri_flpop_co'], pop_flow_2024_df['sat_flpop_co'],
     pop_flow_2024_df['sun_flpop_co']],
    default=0
)

tot_pop = pop_flow_2024_df['tot_flpop_co'].fillna(0)
valid = tot_pop > 0

# 최종 유동인구 값 도출
est_tot_flwpop = np.zeros(len(pop_flow_2024_df))
est_tot_flwpop[valid] = (time_pop[valid] / div[valid]) * (day_pop[valid] / tot_pop[valid]) / 13
pop_flow_2024_df['flwpop_tot'] = est_tot_flwpop

# 연령별 분배
age_map = {
    'agrde_10_flpop_co': 'flwpop_10s', 'agrde_20_flpop_co': 'flwpop_20s', 'agrde_30_flpop_co': 'flwpop_30s',
    'agrde_40_flpop_co': 'flwpop_40s', 'agrde_50_flpop_co': 'flwpop_50s', 'agrde_60_above_flpop_co': 'flwpop_60up'
}
for origin, new in age_map.items():
    pop_flow_2024_df[new] = np.where(valid, est_tot_flwpop * (pop_flow_2024_df[origin] / tot_pop), 0)


# ==========================================
# 최종 데이터셋 완성
# ==========================================
pop_master_2024_df = pd.merge(
    pop_flow_2024_df, pop_living_2024_df,
    on=['date_str', 'hour_str', 'district_name'], how='left'
)

pop_cols = [
    'datetime', 'district_name',
    'flwpop_tot', 'flwpop_10s', 'flwpop_20s', 'flwpop_30s', 'flwpop_40s', 'flwpop_50s', 'flwpop_60up',
    'lvgpop_tot', 'lvgpop_10s', 'lvgpop_20s', 'lvgpop_30s', 'lvgpop_40s', 'lvgpop_50s', 'lvgpop_60up'
]
pop_master_2024_df = pop_master_2024_df[pop_cols]

pop_cols = ['flwpop_tot', 'flwpop_10s', 'flwpop_20s', 'flwpop_30s', 'lvgpop_50s']

for col in pop_cols:
    pop_master_2024_df[col] = pop_master_2024_df.groupby('district_name')[col].transform(
        lambda x: x.fillna(x.median())
    )

print("========== 인구 데이터 전처리 및 병합 완료 ==========")

## 피처 엔지니어링 및 통합 (Feature Engineering)

### 수요 예측용 마스터

In [ ]:
# ==========================================
# 수요 예측용 마스터 데이터 병합
# ==========================================
print("\n========== 마스터 데이터 병합 시작 ==========")

# 1. API로 수집한 2023년 12월 마지막 주 데이터와 2024년 원본 데이터를 위아래로 결합
# (이때 두 데이터프레임의 컬럼명이 완전히 동일해야 합니다!)
rent_history_df = pd.concat(
    [rent_history_2023_df, rent_history_2024_df],
    ignore_index=True
)

rent_history_df['station_id'] = rent_history_df['station_id'].astype(str).str.strip()

# 2. 위치 데이터 병합 (이제 2024_df 대신 combined_df를 기준으로 병합합니다)
demand_predict_master_2024_df = pd.merge(
    rent_history_df,
    station_loc_df[['station_id', 'district', 'lat', 'lon']],
    on='station_id',
    how='left'
)

# 인프라 특성 병합
infra_cols = [
    'station_id', 'subway_cnt_300m', 'biz_cnt_300m', 'edu_cnt_500m',
    'park_cnt_500m', 'river_cnt_1km', 'dist_subway', 'dist_river'
]
demand_predict_master_2024_df = pd.merge(
    demand_predict_master_2024_df,
    infra_master_df[infra_cols],
    on='station_id',
    how='left'
)

# 기상 데이터 병합
demand_predict_master_2024_df['datetime_hr'] = pd.to_datetime(demand_predict_master_2024_df['datetime_hr'])
env_master_2024_df['measure_date'] = pd.to_datetime(env_master_2024_df['measure_date'])

demand_predict_master_2024_df = pd.merge(
    demand_predict_master_2024_df,
    env_master_2024_df[['measure_date', 'region_name', 'temperature', 'precipitation', 'snowfall', 'pm10']],
    left_on=['datetime_hr', 'district'],
    right_on=['measure_date', 'region_name'],
    how='left'
)
demand_predict_master_2024_df.drop(columns=['measure_date', 'region_name'], inplace=True)

# 인구 데이터 병합
pop_master_2024_df['datetime'] = pd.to_datetime(pop_master_2024_df['datetime'])
demand_predict_master_2024_df = pd.merge(
    demand_predict_master_2024_df,
    pop_master_2024_df,
    left_on=['datetime_hr', 'district'],
    right_on=['datetime', 'district_name'],
    how='left'
)
demand_predict_master_2024_df.drop(columns=['datetime', 'district_name'], inplace=True)

# 휴일 데이터 병합
demand_predict_master_2024_df['temp_date'] = demand_predict_master_2024_df['datetime_hr'].dt.date
korea_holidays_df['holiday_date'] = pd.to_datetime(korea_holidays_df['holiday_date']).dt.date

demand_predict_master_2024_df = pd.merge(
    demand_predict_master_2024_df,
    korea_holidays_df[['holiday_date', 'holiday_name']],
    left_on='temp_date',
    right_on='holiday_date',
    how='left'
)

# 파생 변수 생성
demand_predict_master_2024_df['is_holiday'] = demand_predict_master_2024_df['holiday_name'].notna().astype(int)
demand_predict_master_2024_df['day_of_week'] = demand_predict_master_2024_df['datetime_hr'].dt.dayofweek
demand_predict_master_2024_df['is_weekend'] = (demand_predict_master_2024_df['day_of_week'] >= 5).astype(int)
demand_predict_master_2024_df['month'] = demand_predict_master_2024_df['datetime_hr'].dt.month
demand_predict_master_2024_df['hour'] = demand_predict_master_2024_df['datetime_hr'].dt.hour

# 불필요한 임시 컬럼 삭제
cols_to_drop = ['temp_date', 'holiday_date', 'holiday_name', 'district']
demand_predict_master_2024_df.drop(columns=cols_to_drop, inplace=True)

# 수치형 데이터 결측치 보완
fill_zero_cols = [
    'precipitation', 'snowfall', 'subway_cnt_300m',
    'biz_cnt_300m', 'edu_cnt_500m', 'park_cnt_500m', 'river_cnt_1km'
]
demand_predict_master_2024_df[fill_zero_cols] = demand_predict_master_2024_df[fill_zero_cols].fillna(0)

# 메모리 정리
del rent_history_2024_df, env_master_2024_df, pop_master_2024_df, infra_master_df
gc.collect()

print("========== 마스터 데이터 병합 완료 ==========")

### 파생 변수 추가

In [ ]:
# ==============================================================================
# 🚀 파생 변수 생성 및 통합 (수연, 은비, 지혜, 덕윤)
# ==============================================================================
print("\n========== 파생 변수 생성 시작 ==========")

# 시계열 연산(lag, rolling 등) 전 필수 정렬
demand_predict_master_2024_df = demand_predict_master_2024_df.sort_values(
    ['station_id', 'datetime_hr']
).reset_index(drop=True)

# 반복 참조용 임시 변수 할당
_hour = demand_predict_master_2024_df['hour']
_month = demand_predict_master_2024_df['month']
_date = demand_predict_master_2024_df['datetime_hr'].dt.date

# ------------------------------------------------------------------
# 1. 수연님 피처 (시간 주기성, 인구/인프라 밀집도 등)
# ------------------------------------------------------------------
demand_predict_master_2024_df['hour_sin'] = np.sin(2 * np.pi * _hour / 24)
demand_predict_master_2024_df['hour_cos'] = np.cos(2 * np.pi * _hour / 24)
demand_predict_master_2024_df['month_sin'] = np.sin(2 * np.pi * _month / 12)
demand_predict_master_2024_df['month_cos'] = np.cos(2 * np.pi * _month / 12)

demand_predict_master_2024_df['is_rush_hour'] = _hour.isin([7, 8, 9, 18, 19, 20]).astype(int)

demand_predict_master_2024_df['pm10_grade'] = pd.cut(
    demand_predict_master_2024_df['pm10'], bins=[-np.inf, 30, 80, 150, np.inf], labels=[1, 2, 3, 4]
).astype(float)

demand_predict_master_2024_df['is_bad_weather'] = (
    (demand_predict_master_2024_df['precipitation'] > 0) |
    (demand_predict_master_2024_df['snowfall'] > 0) |
    (demand_predict_master_2024_df['pm10_grade'] >= 3)
).astype(int)

flow_to_living_ratio = demand_predict_master_2024_df['flwpop_tot'] / demand_predict_master_2024_df['lvgpop_tot']
demand_predict_master_2024_df['flow_to_living_ratio'] = flow_to_living_ratio.replace([np.inf, -np.inf], np.nan).fillna(0)

demand_predict_master_2024_df['infra_density_score'] = (
    demand_predict_master_2024_df['subway_cnt_300m'] + demand_predict_master_2024_df['biz_cnt_300m'] +
    demand_predict_master_2024_df['edu_cnt_500m'] + demand_predict_master_2024_df['park_cnt_500m'] +
    demand_predict_master_2024_df['river_cnt_1km']
)

is_late_night = _hour.isin([23, 0, 1]).astype(int)
demand_predict_master_2024_df['subway_last_mile_synergy'] = demand_predict_master_2024_df['subway_cnt_300m'] * is_late_night

daily_avg_flwpop = demand_predict_master_2024_df.groupby(['station_id', _date])['flwpop_tot'].transform('mean')
pop_spike_ratio = demand_predict_master_2024_df['flwpop_tot'] / daily_avg_flwpop
demand_predict_master_2024_df['pop_spike_ratio'] = pop_spike_ratio.replace([np.inf, -np.inf], np.nan).fillna(0)

demand_predict_master_2024_df['population_dynamic_flux'] = demand_predict_master_2024_df.groupby('station_id')['lvgpop_tot'].diff().fillna(0)

# ------------------------------------------------------------------
# 2. 은비님 피처 (한강공원 시너지)
# ------------------------------------------------------------------
demand_predict_master_2024_df['is_riverside_park'] = (
    (demand_predict_master_2024_df['park_cnt_500m'] > 0) &
    (demand_predict_master_2024_df['river_cnt_1km'] > 0)
).astype(int)

# ------------------------------------------------------------------
# 3. 지혜님 피처 (기온 및 연휴 이벤트 플래그 + day_type 추가)
# ------------------------------------------------------------------
demand_predict_master_2024_df['is_extreme_temp'] = (
    (demand_predict_master_2024_df['temperature'] <= 0) |
    (demand_predict_master_2024_df['temperature'] >= 30)
).astype(int)

demand_predict_master_2024_df['is_season_change'] = _month.isin([3, 5, 9, 11]).astype(int)

demand_predict_master_2024_df['is_long_weekend'] = (
    demand_predict_master_2024_df['is_holiday'] & demand_predict_master_2024_df['is_weekend']
).astype(int)

# 💡day_type 변수 생성: 평일(0), 주말(1), 휴일(2)
demand_predict_master_2024_df['day_type'] = np.where(
    demand_predict_master_2024_df['is_holiday'] == 1, 2,
    np.where(demand_predict_master_2024_df['is_weekend'] == 1, 1, 0)
)

# ------------------------------------------------------------------
# 4. 덕윤님 피처 (인프라 시너지)
# ------------------------------------------------------------------
demand_predict_master_2024_df['is_biz_hour'] = (
    (demand_predict_master_2024_df['hour'].between(9, 18)) &
    (demand_predict_master_2024_df['is_weekend'] == 0)
).astype(int)

demand_predict_master_2024_df['synergy_biz_weekday'] = (
    demand_predict_master_2024_df['biz_cnt_300m'] * demand_predict_master_2024_df['is_biz_hour']
)

demand_predict_master_2024_df['leisure_infra_cnt'] = (
    demand_predict_master_2024_df['park_cnt_500m'] + demand_predict_master_2024_df['river_cnt_1km']
)

demand_predict_master_2024_df['synergy_leisure_weekend'] = (
    demand_predict_master_2024_df['leisure_infra_cnt'] * demand_predict_master_2024_df['is_weekend']
)

# ------------------------------------------------------------------
# 5. 💡 [최종] 시계열 타겟 피처 통합 (Time-Shift Join 방식)
# 모든 이가 빠진 데이터에 대응 가능한 시간 축 기준 병합
# ------------------------------------------------------------------
rent_target_cols = ['general_rent_cnt', 'sprout_rent_cnt', 'general_rtn_cnt', 'sprout_rtn_cnt']

# 과거 시간(h) 목록 (3시간 이동평균을 위해 1, 2, 3 필요)
# 24시간, 168시간은 그대로 유지
shifts = [1, 2, 3, 24, 168]

# 1. 과거 데이터 베이스 테이블 생성
lag_base_df = demand_predict_master_2024_df[['station_id', 'datetime_hr'] + rent_target_cols].copy()

# 2. 각 shift 간격마다 데이터를 미래로 밀어서 병합
for h in shifts:
    temp_df = lag_base_df.copy()
    temp_df['datetime_hr'] += pd.Timedelta(hours=h)

    # 컬럼명 자동 생성 (예: general_rent_cnt_lag1h)
    new_cols = ['station_id', 'datetime_hr'] + [f'{col}_lag{h}h' for col in rent_target_cols]
    temp_df.columns = new_cols

    # 시간과 대여소를 기준으로 Left Join
    demand_predict_master_2024_df = pd.merge(
        demand_predict_master_2024_df, temp_df,
        on=['station_id', 'datetime_hr'], how='left'
    )

# 3. 결측치 처리 (과거 데이터가 없는 경우 0)
all_lag_cols = [f'{col}_lag{h}h' for h in shifts for col in rent_target_cols]
demand_predict_master_2024_df[all_lag_cols] = demand_predict_master_2024_df[all_lag_cols].fillna(0)

# 4. 이동평균 및 표준편차 연산 (시간 기준)
ts_feature_cols = []
for col in rent_target_cols:
    l1, l2, l3 = f'{col}_lag1h', f'{col}_lag2h', f'{col}_lag3h'

    # 3시간 이동평균
    roll3_col = f'{col}_roll3h_mean'
    demand_predict_master_2024_df[roll3_col] = demand_predict_master_2024_df[[l1, l2, l3]].mean(axis=1)

    # 3시간 이동 표준편차
    roll3_std_col = f'{col}_rolling_std_3h'
    demand_predict_master_2024_df[roll3_std_col] = demand_predict_master_2024_df[[l1, l2, l3]].std(axis=1).fillna(0)

    # 리스트에 최종 사용 피처 추가 (lag2h, lag3h는 리스트 제외하여 모델링에서 제외)
    ts_feature_cols.extend([l1, f'{col}_lag24h', roll3_col, f'{col}_lag168h', roll3_std_col])

# ------------------------------------------------------------------
# 4-1. [피드백 반영] 24시간 rolling 평균 · 표준편차 추가
# station_id 그룹 + 시간순 정렬(상단에서 이미 완료)을 유지한 채,
# 현재 시점을 제외한(shift(1)) 과거 24시간 구간의 이동평균/표준편차를 계산
# ------------------------------------------------------------------
for col in rent_target_cols:
    roll24_mean_col = f'{col}_roll24h_mean'
    roll24_std_col = f'{col}_roll24h_std'

    demand_predict_master_2024_df[roll24_mean_col] = (
        demand_predict_master_2024_df.groupby('station_id')[col]
        .transform(lambda s: s.shift(1).rolling(window=24, min_periods=1).mean())
    )
    demand_predict_master_2024_df[roll24_std_col] = (
        demand_predict_master_2024_df.groupby('station_id')[col]
        .transform(lambda s: s.shift(1).rolling(window=24, min_periods=1).std())
        .fillna(0)
    )

    ts_feature_cols.extend([roll24_mean_col, roll24_std_col])

# ------------------------------------------------------------------
# 메모리 최적화 및 마무리
# ------------------------------------------------------------------
if 'downcast_numeric' in globals():
    demand_predict_master_2024_df = downcast_numeric(demand_predict_master_2024_df)

del _hour, _month, _date, flow_to_living_ratio, daily_avg_flwpop, pop_spike_ratio
gc.collect()

# ------------------------------------------------------------------
# 최종 학습을 위한 FEATURE_COLUMNS 통합 선언
# ------------------------------------------------------------------
base_features = [
    'lat', 'lon', 'subway_cnt_300m', 'biz_cnt_300m', 'edu_cnt_500m', 'park_cnt_500m', 'river_cnt_1km',
    'dist_subway', 'dist_river', 'temperature', 'precipitation', 'snowfall', 'pm10',
    'flwpop_10s', 'flwpop_20s', 'flwpop_30s', 'flwpop_40s', 'flwpop_50s', 'flwpop_60up',
    'lvgpop_10s', 'lvgpop_20s', 'lvgpop_30s', 'lvgpop_40s', 'lvgpop_50s', 'lvgpop_60up',
    'is_holiday', 'day_of_week', 'is_weekend', 'month', 'hour'
]
suyeon_features = [
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'is_rush_hour', 'is_bad_weather', 'pm10_grade',
    'flow_to_living_ratio', 'infra_density_score', 'subway_last_mile_synergy', 'pop_spike_ratio', 'population_dynamic_flux'
]
eunbi_features = ['is_riverside_park']
jihye_features = ['is_extreme_temp', 'is_season_change', 'is_long_weekend', 'day_type']
deokyun_features = ['is_biz_hour', 'synergy_biz_weekday', 'leisure_infra_cnt', 'synergy_leisure_weekend']

# 모든 리스트 최종 합산 (ts_feature_cols 20개가 여기서 자동으로 달라붙습니다)
FEATURE_COLUMNS = base_features + suyeon_features + eunbi_features + jihye_features + deokyun_features + ts_feature_cols

demand_predict_master_2024_df = demand_predict_master_2024_df[
    demand_predict_master_2024_df['datetime_hr'] >= '2024-01-01'
].reset_index(drop=True)

print(f"========== 팀원 통합 파생 변수 생성 완료 (총 {len(FEATURE_COLUMNS)}개 피처) ==========")

In [ ]:
display(demand_predict_master_2024_df.head(20))

## 모델링 및 실험 관리 (Modeling & MLOps)

### 시계열 학습, 검증, 테스트셋 분할

In [ ]:
# ==========================================
# 1. 시계열 순서 정렬 및 분할 인덱스 계산
# ==========================================
TARGET_COLUMNS = ['general_rent_cnt', 'sprout_rent_cnt', 'general_rtn_cnt', 'sprout_rtn_cnt']

# 시계열 순서 정렬 (매우 중요)
df_sorted = demand_predict_master_2024_df.sort_values("datetime_hr").reset_index(drop=True)

# 분할 인덱스 선 계산 (6:2:2)
n = len(df_sorted)
train_end = int(n * 0.60)
val_end   = int(n * 0.80)

# ==========================================
# 2. 👨‍💻 덕윤님 피처 (타겟 인코딩 - 데이터 누수 차단 완벽 적용)
# ==========================================
print("========== 타겟 인코딩 (station_dow_hour_mean) 생성 중 ==========")

# 오직 Train 데이터만 사용하여 평균을 구함 (미래 데이터 참조 방지)
train_only_df = df_sorted.iloc[:train_end].copy()
group_cols = ['station_id', 'day_of_week', 'hour']  # [피드백 반영] 요일 단위로 세분화

encoded_feature_names = []

for target in TARGET_COLUMNS:
    encoded_col_name = f'{target}_station_dow_hour_mean'
    encoded_feature_names.append(encoded_col_name)

    # 1. Train 기준으로 그룹 평균 계산
    target_mean = train_only_df.groupby(group_cols)[target].mean().reset_index()
    target_mean.rename(columns={target: encoded_col_name}, inplace=True)

    # 2. 전체 데이터셋에 매핑
    df_sorted = pd.merge(df_sorted, target_mean, on=group_cols, how='left')

    # 3. 결측치 보완 1단계 (요일 조건 제외, 대여소+시간 기준)
    fallback_cols = ['station_id', 'hour']
    fallback_mean = train_only_df.groupby(fallback_cols)[target].mean().reset_index()
    fallback_mean.rename(columns={target: 'fallback_mean'}, inplace=True)

    df_sorted = pd.merge(df_sorted, fallback_mean, on=fallback_cols, how='left')
    df_sorted[encoded_col_name] = df_sorted[encoded_col_name].fillna(df_sorted['fallback_mean'])
    df_sorted.drop(columns=['fallback_mean'], inplace=True)

    # 4. 결측치 보완 2단계 (전체 평균)
    overall_mean = train_only_df[target].mean()
    df_sorted[encoded_col_name] = df_sorted[encoded_col_name].fillna(overall_mean)

print("========== 타겟 인코딩 생성 완료 ==========")

# ==========================================
# 3. 피처 및 타깃 정의 (최종 X, Y)
# ==========================================
# 🚨 윗 셀에서 선언된 FEATURE_COLUMNS 리스트에 타겟 인코딩 변수명들을 합쳐줍니다.
FEATURE_COLUMNS.extend(encoded_feature_names)

# 완성된 df_sorted에서 최종 X, Y 추출
X = df_sorted[FEATURE_COLUMNS]
Y = df_sorted[TARGET_COLUMNS]

# 모델 호환성을 위한 컬럼명 문자열 변환 (LightGBM, XGBoost 등 에러 방지)
X.columns = [str(col) for col in X.columns]

# ==========================================
# 4. 시계열 데이터 분할 (6:2:2)
# ==========================================
X_train, Y_train = X.iloc[:train_end], Y.iloc[:train_end]
X_val, Y_val     = X.iloc[train_end:val_end], Y.iloc[train_end:val_end]
X_test, Y_test   = X.iloc[val_end:], Y.iloc[val_end:]

# ==========================================
# 5. 데이터 분할 결과 요약 테이블 생성
# ==========================================
split_summary = pd.DataFrame({
    "Dataset": ["Train", "Val", "Test"],
    "Rows": [len(X_train), len(X_val), len(X_test)],
    "Start Date": [
        df_sorted['datetime_hr'].iloc[0],
        df_sorted['datetime_hr'].iloc[train_end],
        df_sorted['datetime_hr'].iloc[val_end]
    ],
    "End Date": [
        df_sorted['datetime_hr'].iloc[train_end-1],
        df_sorted['datetime_hr'].iloc[val_end-1],
        df_sorted['datetime_hr'].iloc[-1]
    ]
})

# 가독성을 위해 행 수(Rows)에 콤마 추가
split_summary["Rows"] = split_summary["Rows"].apply(lambda x: f"{x:,}")

print("========== 데이터 분할 요약 ==========")
display(split_summary)

# 메모리 정리
del df_sorted, demand_predict_master_2024_df
gc.collect()

### 베이스라인 모델 학습 및 MLflow 실험 관리

In [ ]:
# ==========================================
# Optuna / TimeSeriesSplit 관련 라이브러리
# ==========================================
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import VotingRegressor, StackingRegressor
optuna.logging.set_verbosity(optuna.logging.WARNING)  # 튜닝 로그 너무 많이 안 찍히게

# ==========================================
# MLflow 환경 설정
# ==========================================
USE_TEAM_SERVER = True
TEAM_SERVER_URI = "http://223.194.48.21:5000"
AUTHOR = "장수연"

mlflow.set_tracking_uri(TEAM_SERVER_URI if USE_TEAM_SERVER else "sqlite:///mlflow_seoul_bike.db")
mlflow.set_experiment("bike_demand_prediction")


# ==========================================
# 평가 지표 함수 (RMSLE 기준, 항상 원본 스케일에서 계산)
# ==========================================
def rmsle(y_true_raw, pred_raw):
    log_y = np.log1p(np.maximum(y_true_raw, 0))
    log_pred = np.log1p(np.maximum(pred_raw, 0))
    return np.sqrt(np.mean((log_y - log_pred) ** 2))

def evaluate_regr(y_true_raw, pred_raw):
    return {
        "rmsle": rmsle(y_true_raw, pred_raw),
        "rmse": np.sqrt(mean_squared_error(y_true_raw, pred_raw)),
        "mae": mean_absolute_error(y_true_raw, pred_raw)
    }

# [피드백 1] RMSLE 평가와 학습 목표를 일치시키기 위한 변환 헬퍼
def to_log1p(y_raw):
    return np.log1p(y_raw)

def to_raw(pred_log):
    # log1p 로 학습했으므로 예측은 expm1 로 복원, 음수는 0으로 clip
    return np.clip(np.expm1(pred_log), 0, None)


# ==========================================
# 모델 정의: 모델명 + trial -> 모델 인스턴스 (기존과 동일)
# ==========================================
def build_model(model_name, trial, SEED=SEED):
    if model_name == "LinearRegression":
        return make_pipeline(StandardScaler(), LinearRegression())

    if model_name == "Ridge":
        return make_pipeline(StandardScaler(), Ridge(alpha=1.0))

    if model_name == "Lasso":
        return make_pipeline(StandardScaler(), Lasso(alpha=0.1))

    if model_name == "ElasticNet":
        return make_pipeline(StandardScaler(), ElasticNet(alpha=0.1, l1_ratio=0.5))

    if model_name == "RandomForest":
        return RandomForestRegressor(n_estimators=50, max_depth=10, random_state=SEED, n_jobs=-1)

    if model_name == "GradientBoosting":
        return GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=SEED)

    if model_name == "LightGBM":
        return LGBMRegressor(n_estimators=100, random_state=SEED, n_jobs=-1, verbosity=-1)

    if model_name == "XGBoost":
        return XGBRegressor(n_estimators=100, random_state=SEED, n_jobs=-1, tree_method='hist')

    raise ValueError(f"알 수 없는 모델명: {model_name}")


MODEL_NAMES = [
    "LinearRegression", "Ridge", "Lasso", "ElasticNet",
    "RandomForest", "GradientBoosting", "LightGBM", "XGBoost"
]

N_TRIALS = 30       # 모델별 Optuna 탐색 횟수 (필요에 맞게 조절하세요)
N_SPLITS = 5        # [피드백 2] TimeSeriesSplit 폴드 수
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

# [피드백 3] 앙상블 base로 사용할 모델 (LightGBM · XGBoost · 선형모델)
ENSEMBLE_BASE_MODELS = ["LightGBM", "XGBoost", "Ridge"]


# ==========================================
# 모델 학습(Optuna + TimeSeriesSplit 튜닝 포함) 및 MLflow 기록
# ==========================================
results = []
tuned_models = {}        # {target_name: {model_name: fitted_model}}
best_params_store = {}   # {target_name: {model_name: best_params}}
run_date = datetime.now().strftime("%Y-%m-%d %H:%M")

for target_name in TARGET_COLUMNS:
    print(f"========== [{target_name}] 모델 학습 시작 ==========")

    y_train_raw = Y_train[target_name]
    y_val_raw = Y_val[target_name]

    # [피드백 1] 타깃을 log1p로 변환해서 학습
    y_train_log = to_log1p(y_train_raw)

    tuned_models[target_name] = {}
    best_params_store[target_name] = {}

    for model_name in MODEL_NAMES:
        run_name = f"{model_name}_{target_name}"

        try:
            if model_name == "LinearRegression":
                # 튜닝할 파라미터가 없으므로 바로 학습
                best_params = {}
                model = build_model(model_name, trial=None)
            else:
                def objective(trial, model_name=model_name):
                    fold_scores = []
                    # [피드백 2] TimeSeriesSplit(expanding window)으로 여러 구간 검증
                    for tr_idx, va_idx in tscv.split(X_train):
                        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
                        y_tr_log = y_train_log.iloc[tr_idx]
                        y_va_raw = y_train_raw.iloc[va_idx]

                        fold_model = build_model(model_name, trial)
                        fold_model.fit(X_tr, y_tr_log)
                        pred_raw = to_raw(fold_model.predict(X_va))
                        fold_scores.append(rmsle(y_va_raw.values, pred_raw))

                    return float(np.mean(fold_scores))

                study = optuna.create_study(direction="minimize")
                study.optimize(objective, n_trials=N_TRIALS)
                best_params = study.best_params

                # 최적 파라미터로 최종 모델 재학습
                best_trial = optuna.trial.FixedTrial(best_params)
                model = build_model(model_name, best_trial)

            # 폴드 튜닝이 끝난 뒤 Train 전체(log1p)로 최종 학습 → Val로 최종 평가
            model.fit(X_train, y_train_log)
            pred_val_raw = to_raw(model.predict(X_val))
            metrics = evaluate_regr(y_val_raw.values, pred_val_raw)

            tuned_models[target_name][model_name] = model
            best_params_store[target_name][model_name] = best_params

            with mlflow.start_run(run_name=run_name):
                mlflow.log_param("target", target_name)
                mlflow.log_param("model", model_name)
                for p_name, p_val in best_params.items():
                    mlflow.log_param(p_name, p_val)
                mlflow.set_tag("author", AUTHOR)
                mlflow.set_tag("run_date", run_date)
                mlflow.log_metrics(metrics)

            results.append({
                "target": target_name,
                "model_name": model_name,
                "rmsle": metrics["rmsle"],
                "rmse": metrics["rmse"],
                "mae": metrics["mae"]
            })
        except Exception as e:
            print(f"[{run_name}] 학습 실패: {e}")

print("========== 개별 모델 학습 완료 ==========")


# ==========================================
# [피드백 6] RegressorChain으로 4개 타깃 간 상관관계 확인
# (개별 모델 학습과 같은 베이스라인 실험 단계에서 함께 진행)
# 체인의 각 단계는 예측(predict) 시 "실제 미래값"이 아니라
# "이전 단계 모델의 예측값"만 다음 단계 입력 피처로 사용하므로
# 미래 정보 누수는 발생하지 않음
# (단, sklearn RegressorChain은 학습(fit) 시에는 이전 타깃의 실제값을
#  다음 타깃 학습 피처로 사용하는 것이 기본 동작입니다 — 표준 설계이며
#  동시점 반납량을 그대로 대여 예측 피처로 노출하는 것과는 다른 개념입니다)
# ==========================================
from sklearn.multioutput import RegressorChain

print("\n========== RegressorChain (4개 타깃 관계 확인) 시작 ==========")

chain_target_order = ['general_rent_cnt', 'general_rtn_cnt', 'sprout_rent_cnt', 'sprout_rtn_cnt']

Y_train_chain_log = Y_train[chain_target_order].apply(np.log1p)
Y_val_chain_raw = Y_val[chain_target_order]

_lgbm_params = best_params_store.get(chain_target_order[0], {}).get("LightGBM", {})
chain_base = LGBMRegressor(**_lgbm_params, random_state=SEED, n_jobs=-1, verbosity=-1)

chain_model = RegressorChain(base_estimator=chain_base, order=list(range(len(chain_target_order))))
chain_model.fit(X_train, Y_train_chain_log)

chain_pred_log = chain_model.predict(X_val)
chain_pred_raw = np.clip(np.expm1(chain_pred_log), 0, None)

for i, target_name in enumerate(chain_target_order):
    metrics = evaluate_regr(Y_val_chain_raw[target_name].values, chain_pred_raw[:, i])

    with mlflow.start_run(run_name=f"RegressorChain_{target_name}"):
        mlflow.log_param("target", target_name)
        mlflow.log_param("model", "RegressorChain")
        mlflow.set_tag("author", AUTHOR)
        mlflow.set_tag("run_date", run_date)
        mlflow.log_metrics(metrics)

    results.append({
        "target": target_name, "model_name": "RegressorChain",
        "rmsle": metrics["rmsle"], "rmse": metrics["rmse"], "mae": metrics["mae"]
    })

print("========== RegressorChain 완료 ==========")


# ==========================================
# 결과 확인 (개별 모델 + RegressorChain)
# ==========================================
results_df = pd.DataFrame(results)
results_df.rename(columns={
    'rmsle': 'RMSLE',
    'rmse': 'RMSE',
    'mae': 'MAE'
}, inplace=True)

display(results_df.sort_values(by=['target', 'RMSLE']))


### 앙상블 베이스라인 (Voting / Stacking) — 팀 공통 시작점 (Floor)

- 이 셀은 팀원들이 함께 앙상블을 발전시킬 때 기준이 되는 **앙상블 Floor**입니다. 개별 모델 Floor와 같은 원칙으로, 이후 팀원들이 시도하는 다양한 앙상블 조합/구조는 이 결과와 비교해서 성능 향상 여부를 판단하면 됩니다.
- `StackingRegressor(cv=TimeSeriesSplit(...))`를 통한 **OOF 예측(누수 차단)** 은 팀원이 다른 base 모델 조합이나 메타모델로 바꿔서 시도하더라도 항상 유지해야 하는 최소 요건입니다 — 선택 사항이 아니라 정답을 맞추기 위한 필수 조건에 가깝습니다.
- 반면 24시간 rolling·요일 기반 타겟 인코딩 같은 피처 확장은 이미 `X_train`/`X_val`(및 아래에서 pickle로 저장되는 공통 데이터셋)에 반영돼 있어서, 팀원이 이 데이터셋을 그대로 불러와 앙상블을 진행하는 한 별도로 신경 쓰지 않아도 자동으로 적용됩니다.


In [ ]:
# ==========================================
# 앙상블 (Voting / Stacking) — [피드백 3, 4]
# LightGBM · XGBoost · 선형모델(Ridge)을 base로, Ridge를 메타모델로 사용
# StackingRegressor(cv=TimeSeriesSplit)로 OOF 예측을 생성해 메타 입력 누수를 차단
# ==========================================
print("\n========== 앙상블 모델 학습 시작 ==========")

for target_name in TARGET_COLUMNS:
    y_train_raw = Y_train[target_name]
    y_val_raw = Y_val[target_name]
    y_train_log = to_log1p(y_train_raw)

    base_estimators = []
    for m_name in ENSEMBLE_BASE_MODELS:
        if m_name not in best_params_store[target_name]:
            continue
        if m_name == "LinearRegression":
            fresh_model = build_model(m_name, trial=None)
        else:
            fixed_trial = optuna.trial.FixedTrial(best_params_store[target_name][m_name])
            fresh_model = build_model(m_name, fixed_trial)
        base_estimators.append((m_name, fresh_model))

    if len(base_estimators) < 2:
        print(f"[{target_name}] 앙상블 base 모델이 부족해 스킵합니다.")
        continue

    # ---- Voting: 기준선 ----
    voting_model = VotingRegressor(estimators=base_estimators, n_jobs=-1)
    voting_model.fit(X_train, y_train_log)
    voting_pred_raw = to_raw(voting_model.predict(X_val))
    voting_metrics = evaluate_regr(y_val_raw.values, voting_pred_raw)

    with mlflow.start_run(run_name=f"Voting_{target_name}"):
        mlflow.log_param("target", target_name)
        mlflow.log_param("model", "Voting")
        mlflow.set_tag("author", AUTHOR)
        mlflow.set_tag("run_date", run_date)
        mlflow.log_metrics(voting_metrics)

    results.append({
        "target": target_name, "model_name": "Voting",
        "rmsle": voting_metrics["rmsle"], "rmse": voting_metrics["rmse"], "mae": voting_metrics["mae"]
    })

    # ---- Stacking: OOF 예측 기반 확장 (누수 차단) ----
    stacking_model = StackingRegressor(
        estimators=base_estimators,
        final_estimator=Ridge(),
        cv=TimeSeriesSplit(n_splits=N_SPLITS),   # [피드백 4] OOF 예측 생성 → 누수 차단
        n_jobs=-1
    )
    stacking_model.fit(X_train, y_train_log)
    stacking_pred_raw = to_raw(stacking_model.predict(X_val))
    stacking_metrics = evaluate_regr(y_val_raw.values, stacking_pred_raw)

    with mlflow.start_run(run_name=f"Stacking_{target_name}"):
        mlflow.log_param("target", target_name)
        mlflow.log_param("model", "Stacking")
        mlflow.set_tag("author", AUTHOR)
        mlflow.set_tag("run_date", run_date)
        mlflow.log_metrics(stacking_metrics)

    results.append({
        "target": target_name, "model_name": "Stacking",
        "rmsle": stacking_metrics["rmsle"], "rmse": stacking_metrics["rmse"], "mae": stacking_metrics["mae"]
    })

print("========== 앙상블 모델 학습 완료 ==========")

results_df = pd.DataFrame(results)
results_df.rename(columns={'rmsle': 'RMSLE', 'rmse': 'RMSE', 'mae': 'MAE'}, inplace=True)
display(results_df.sort_values(by=['target', 'RMSLE']))


### 모델 및 데이터셋 아카이빙

In [ ]:
# ==========================================
# 데이터셋 저장
# ==========================================
save_dir = "D:/seoul_bike/models_pkl/train_pkl"
os.makedirs(save_dir, exist_ok=True)

datasets = {
    "X_train.pkl": X_train, "Y_train.pkl": Y_train,
    "X_val.pkl": X_val,     "Y_val.pkl": Y_val,
    "X_test.pkl": X_test,   "Y_test.pkl": Y_test
}

for filename, data in datasets.items():
    joblib.dump(data, os.path.join(save_dir, filename))

print(f"========== [{save_dir}] 공통 데이터셋 저장 완료 ==========")